# Classical features and SVM baselines

After waveform, FFT and envelope inspection, the research asked whether four simple time-domain features already separated the selected recordings. RMS, absolute peak, Pearson kurtosis and crest factor describe amplitude and distribution shape; their predictive success need not imply fault-specific physics.

Default cells show a labeled synthetic feature illustration and six saved historical SVM observations. They fit no model. Raw data are required only for opt-in new fitting. Set `CWRU_DATA_ROOT` and optionally `CWRU_ARTIFACT_ROOT` before launching Jupyter. See [signal exploration](01_signal_exploration.ipynb), the [experiment map](../docs/exploration.md), and the [classical evidence extract](../results/historical/classical_svm_evidence.json).

In [1]:
from pathlib import Path
import os, sys, json
candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'src/cwru_bearing').is_dir()), None)
if REPO is None:
    raise RuntimeError('Open this notebook from the repository checkout or its notebooks directory.')
sys.path.insert(0, str(REPO / 'src'))
DATA_ROOT = Path(os.environ['CWRU_DATA_ROOT']) if os.environ.get('CWRU_DATA_ROOT') else None
ARTIFACT_ROOT = Path(os.environ.get('CWRU_ARTIFACT_ROOT', str(REPO / 'runs')))
print('Data configured:', DATA_ROOT is not None)
print('No training is enabled by default.')

Data configured: False
No training is enabled by default.


## What the features describe

Each non-overlapping 2048-sample window yields four values. RMS describes overall amplitude, absolute peak captures the largest excursion, Pearson kurtosis (`fisher=False`) measures distribution shape, and crest factor compares peak with RMS. The RBF SVM standardizes its inputs using training data only.

The following sine wave is a numerical illustration of the feature API, not a CWRU recording or an experimental result. Its regular waveform should not be interpreted as a bearing-fault example.


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
from cwru_bearing.classical import windows, features
synthetic = np.sin(np.arange(8192) * 0.08)  # Illustration only, not CWRU evidence.
illustrative_features = np.asarray([features(w) for w in windows(synthetic)])
display(pd.DataFrame(illustrative_features, columns=['RMS', 'Absolute peak', 'Pearson kurtosis', 'Crest factor']).rename_axis('Synthetic window'))

,RMS,Absolute peak,Pearson kurtosis,Crest factor
Synthetic window,,,,
0,0.706191,1.0,1.503477,1.416047
1,0.706891,1.0,1.499514,1.414645
2,0.707772,1.0,1.498028,1.412884
3,0.708091,1.0,1.498446,1.412247


## Three historical experimental definitions

1. **Fixed-size cross-load (E79–85):** healthy and .007-inch faults; loads 0/1/2 train, load 3 test. Four-feature and RMS-only accuracy were both 100% on 415 windows.
2. **Joint load/size transfer (E86–95):** healthy and .007-inch faults at loads 0/1/2 train; healthy and .014-inch faults at load 3 test. Saved accuracy was 58.70% with four features and 68.36% with RMS only on 414 windows. This changes load and size together. Healthy windows account for 237/414 tests; the aggregate score hides much weaker faulty-class behavior.
3. **Fault-only size transfer (E100–103):** .007-inch faults train and .014-inch faults test over all four loads, with no healthy class. Saved accuracy was 43.79% with four features and 32.63% with RMS alone on 708 windows.

These are different protocols and class balances. They are not a controlled ranking of methods or shifts. The public evidence extract contains the exact saved accuracy values, matrices and original cell references; the full original notebook remains private. No fitting is performed by the table cell.

In the joint-shift matrices, all 237 healthy windows were correct. Faulty windows alone contributed only 6/177 correct with four features and 46/177 with RMS alone. These counts are derived from the saved matrices, not from new predictions.

In [3]:
historical = json.loads((REPO / 'results/historical/classical_svm_evidence.json').read_text(encoding='utf-8'))
experiment_names = {'fixed_size_cross_load': 'Fixed-size cross-load',
                    'joint_load_size_transfer': 'Joint load/size transfer',
                    'fault_only_size_transfer': 'Fault-only size transfer'}
display(pd.DataFrame([
    {'Experiment': experiment_names[observation['id'].rsplit('_four_features', 1)[0].rsplit('_rms_only', 1)[0]],
     'SVM features': 'RMS only' if observation['features'] == ['RMS'] else 'Four features',
     'Test windows': sum(map(sum, observation['confusion_matrix'])),
     'Accuracy (%)': 100 * observation['accuracy'], 'Original cell': f"E{observation['source_cell']}"}
    for observation in sorted(historical['observations'], key=lambda item: item['source_cell'])
]).round(2))

,Experiment,SVM features,Test windows,Accuracy (%),Original cell
0,Fixed-size cross-load,Four features,415,100.00,E82
1,Fixed-size cross-load,RMS only,415,100.00,E85
2,Joint load/size transfer,Four features,414,58.70,E93
3,Joint load/size transfer,RMS only,414,68.36,E95
4,Fault-only size transfer,Four features,708,43.79,E102
5,Fault-only size transfer,RMS only,708,32.63,E103


Healthy signals were acquired at documented 48 kHz but historically processed as 12 kHz. Raw feature values survive a rate-label correction, while physical window duration and acquisition bandwidth differ. Rate harmonization and new classical comparisons require a separately defined and evaluated protocol.

## Feature shifts explain why the protocol matters

E94 compared training/test feature means, E96–99 plotted their distributions, and E104–107 grouped RMS and kurtosis by fault size and load. They show that selected fault conditions do not preserve the same amplitude statistics across sizes. These are descriptive observations on the inspected data, not independent tests of a physical mechanism. The saved E94 table below mixes load and size changes by its original definition.

In [4]:
signal_evidence = json.loads((REPO / 'results/historical/signal_exploration_evidence.json').read_text(encoding='utf-8'))
print(signal_evidence['outputs']['94'])

TRAIN mean features:
              RMS    Peak  Kurtosis  CrestFactor
label                                           
Ball       0.1417  0.4934    2.9129       3.4826
Healthy    0.0670  0.2197    2.8792       3.2806
InnerRace  0.2946  1.4475    5.4980       4.9149
OuterRace  0.6104  3.0167    7.6864       4.9466

TEST mean features:
              RMS    Peak  Kurtosis  CrestFactor
label                                           
Ball       0.1297  0.8398    7.9904       6.2151
Healthy    0.0659  0.2216    2.9461       3.3637
InnerRace  0.1805  1.5725   17.7784       8.7003
OuterRace  0.0945  0.3838    3.4416       4.0256



## Optional baseline fitting

Enable `RUN_CLASSICAL` only after configuring the raw recordings and choosing a fresh output directory. This reconstructs the fixed-size cross-load, joint load/size-transfer, and fault-only size-transfer investigations with the historical windows, features, StandardScaler, and default RBF SVC. Validation checks the 28 healthy, .007-inch, and .014-inch recordings used by these classical experiments. New results are written separately from the saved findings; enabling this cell does not recover the historical session.


In [5]:
RUN_CLASSICAL = False
if RUN_CLASSICAL:
    if DATA_ROOT is None:
        raise RuntimeError('Configure CWRU_DATA_ROOT first.')
    from cwru_bearing.classical import run_baselines
    output = ARTIFACT_ROOT / 'classical' / 'run01'
    if output.exists():
        raise FileExistsError('Choose a new classical output directory.')
    results = run_baselines(DATA_ROOT)
    output.mkdir(parents=True)
    (output / 'results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
    print(results)
else:
    print('Classical fitting disabled; historical evidence was not replaced.')

Classical fitting disabled; historical evidence was not replaced.


[Previous: signal exploration](01_signal_exploration.ipynb) | [Project overview](../README.md) | [Next: STFT exploration](03_stft_exploration.ipynb)